# 01 - Locate / Validate Dataset

This notebook prepares the **IU X-Ray (Indiana University Chest X-Ray)** dataset for the
benchmark, using the public Kaggle dataset:

https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university

**On Kaggle:** click *Add Input* on the right-hand panel, search for
`raddar/chest-xrays-indiana-university`, and attach it. `datasets/iu_xray.py` auto-detects
the mounted path (`/kaggle/input/chest-xrays-indiana-university` or
`/kaggle/input/datasets/raddar/chest-xrays-indiana-university`), so no manual path editing
is required. If you've placed the dataset somewhere else, set `dataset.root_dir` in
`configs/default.yaml`.

This notebook does not assume a pre-processed `reports.csv` exists -- it works directly
against the raw `indiana_reports.csv` + `indiana_projections.csv` files, and:

1. Checks the dataset is available (and shows exactly where it looked, if not).
2. Validates both CSVs have the expected columns.
3. Prints dataset statistics (studies, images, missing report text).
4. Displays a few sample images with their reports.
5. Runs `datasets.load_dataset()` end-to-end to confirm the benchmark loader itself works.

In [ ]:
import sys

REPO_DIR = "/kaggle/working/medical-report-benchmark"
import os
if not os.path.exists(REPO_DIR):
    # Not running on Kaggle with the repo attached under this path -- fall back to
    # the current working directory (e.g. running the notebook locally from the repo root).
    REPO_DIR = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'datasets')) else os.path.dirname(os.getcwd())

sys.path.insert(0, REPO_DIR)
print('Using repo dir:', REPO_DIR)

In [ ]:
# Step 1: check dataset availability using the same auto-detection logic the
# benchmark loader itself uses -- no manual path editing needed.
from utils.io import load_config
from datasets.iu_xray import (
    _candidate_roots,
    _find_legacy_reports_csv,
    _locate_dataset_root,
    _locate_images_dir,
    REPORTS_CSV_NAME,
    PROJECTIONS_CSV_NAME,
)

config = load_config('configs/default.yaml')
ds_cfg = config['dataset']
roots = _candidate_roots(ds_cfg)
print('Checked candidate roots:')
for r in roots:
    print(' -', r, '(exists)' if r.exists() else '(missing)')

legacy_csv = _find_legacy_reports_csv(ds_cfg, roots)
if legacy_csv is not None:
    print(f'\nFound legacy pre-processed reports.csv at: {legacy_csv}')
    print('The loader will use this instead of the raw Kaggle CSVs.')
    dataset_root = None
    images_dir = None
else:
    dataset_root = _locate_dataset_root(roots)
    images_dir = _locate_images_dir(ds_cfg, dataset_root)
    print(f'\nFound raw Kaggle dataset at: {dataset_root}')
    print(f'Images directory: {images_dir}')
    print(f'{REPORTS_CSV_NAME} exists:', (dataset_root / REPORTS_CSV_NAME).exists())
    print(f'{PROJECTIONS_CSV_NAME} exists:', (dataset_root / PROJECTIONS_CSV_NAME).exists())

In [ ]:
# Step 2: validate the raw CSVs directly (skipped if a legacy reports.csv was found above).
import pandas as pd

if dataset_root is not None:
    reports_df = pd.read_csv(dataset_root / REPORTS_CSV_NAME)
    projections_df = pd.read_csv(dataset_root / PROJECTIONS_CSV_NAME)

    required_report_cols = {'uid', 'findings', 'impression'}
    required_projection_cols = {'uid', 'filename'}
    missing_report_cols = required_report_cols - set(reports_df.columns)
    missing_projection_cols = required_projection_cols - set(projections_df.columns)

    assert not missing_report_cols, f'indiana_reports.csv missing columns: {missing_report_cols}'
    assert not missing_projection_cols, f'indiana_projections.csv missing columns: {missing_projection_cols}'

    print('indiana_reports.csv columns:', list(reports_df.columns))
    print('indiana_projections.csv columns:', list(projections_df.columns))
    print('CSV schema OK.')
else:
    print('Using legacy reports.csv; raw CSV validation skipped.')

In [ ]:
# Step 3: dataset statistics.
if dataset_root is not None:
    n_studies = reports_df['uid'].nunique()
    n_images = len(projections_df)
    n_missing_findings = reports_df['findings'].isna().sum()
    n_missing_impression = reports_df['impression'].isna().sum()
    n_missing_both = (reports_df['findings'].isna() & reports_df['impression'].isna()).sum()

    print(f'Studies (unique uid):       {n_studies}')
    print(f'Images (projection rows):   {n_images}')
    print(f'Avg images per study:       {n_images / n_studies:.2f}')
    print(f'Studies missing findings:   {n_missing_findings}')
    print(f'Studies missing impression: {n_missing_impression}')
    print(f'Studies missing both:       {n_missing_both} (these are dropped by the loader)')
    print()
    print('Projection type counts:')
    print(projections_df['projection'].value_counts())

In [ ]:
# Step 4: display a few sample images with their merged report text.
import matplotlib.pyplot as plt
from PIL import Image

if dataset_root is not None:
    sample = projections_df.merge(reports_df[['uid', 'findings', 'impression']], on='uid').head(4)

    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img_path = images_dir / row['filename']
        if img_path.exists():
            ax.imshow(Image.open(img_path).convert('L'), cmap='gray')
        else:
            ax.text(0.5, 0.5, 'image not found', ha='center', va='center')
        ax.set_title(row['filename'], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    for _, row in sample.head(2).iterrows():
        print(f"--- uid={row['uid']} ({row['filename']}) ---")
        print('Findings:  ', row['findings'])
        print('Impression:', row['impression'])
        print()
else:
    print('Using legacy reports.csv; raw sample preview skipped (see loader output below).')

In [ ]:
# Step 5: run the actual benchmark loader end-to-end to confirm everything works together.
from datasets import load_dataset

df = load_dataset(config)
print(f'Loaded {len(df)} samples via datasets.load_dataset().')
assert list(df.columns) == ['image_id', 'image_path', 'ground_truth_report']
assert len(df) > 0, 'Loader returned zero samples -- check dataset attachment / paths above.'
df.head()